In [ ]:
# NOTE: NOTEBOOK PARAMETERS
setup = 'cls_loc_models'
selected_ibtracs = 'ibtracs_main-tracks_6h_1980-2021_TS-ET-SS.csv'

# years and months of the Test Setup
years = [i for i in range(1980,2020)]
months = [8]

# lat and lon ranges of the domain
lon_range = [100, 320]
lat_range = [0,70]

# minimum number of consecutive detection at 6h lead time to consider the track true
min_track_count = 12

# info
remove_atlantic = True
remove_short_tracks = True

# maximum distance to consider true the match
max_track_distance_matching = 300.0

# whether to store the figures or not
save_figures = True

In [ ]:
from cartopy.mpl.ticker import (LongitudeFormatter, LatitudeFormatter)
import matplotlib.transforms as transforms
import matplotlib.ticker as mticker
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import cartopy.feature as cf
import cartopy.crs as ccrs

import pandas as pd
import numpy as np
import glob
import os
import sys

sys.path.append('../resources/library/tropical_cyclone')
import dynamicopy

from tropical_cyclone.cyclone import (
    filter_basin, 
    filter_tracks, 
    compute_pod_and_far, 
    plot_storm_track_density, 
    compute_storm_transits, 
    signif, 
    plot_pod_and_far_multi_trackers, 
    plot_tracks, 
)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# main dirs
data_src = 'PATH-TO/data'
bourdin_trackers_src = 'PATH-TO/zenodo_bourdin/trackers'
inference_src = os.path.join(data_src, 'inference')

# trackers source
ibtracs_src = os.path.join(data_src, 'ibtracs', 'filtered', selected_ibtracs)
owz_src = os.path.join(bourdin_trackers_src, 'OWZ', 'tracks', 'ERA5_OWZ.csv')
track_src = os.path.join(bourdin_trackers_src, 'TRACK', 'ERA5_TRACK.csv')
uz_src = os.path.join(bourdin_trackers_src, 'UZ', 'tracks', 'ERA5_UZ.csv')
cnrm_src = os.path.join(bourdin_trackers_src, 'CNRM', 'tracks_vor15_res5_t1_pt-1_pw5_rel25.csv')

# our model
hybrid_ml_trackers_src = os.path.join(data_src, f'inference/{setup}/*/tracking.csv')
hybrid_fnames = sorted(glob.glob(hybrid_ml_trackers_src))

# directory to store images
out_dir = f'./images/{setup}/comparison_table'

In [ ]:
os.makedirs(out_dir, exist_ok=True)

In [ ]:
hybrid_fnames

In [ ]:
hybrid_key_renames = {
    f'localization_model-classification_model': 'loc_rv_msl-cls_rv_msl', 
}

In [ ]:
ibtracs = pd.read_csv(ibtracs_src, index_col=0).rename(columns={'ISO_TIME':'time','LAT':'lat','LON':'lon','SID':'track_id'})[['time','lat','lon','track_id','NATURE','WMO_WIND']]
# ibtracs = dynamicopy.load_ibtracs()
owz = pd.read_csv(owz_src).rename(columns={' year':'year', ' month':'month', ' day':'day', ' hour':'hour',' lon':'lon',' lat':'lat'})
track = pd.read_csv(track_src, index_col=0)
uz = pd.read_csv(uz_src).rename(columns={' year':'year', ' month':'month', ' day':'day', ' hour':'hour',' lon':'lon',' lat':'lat'})
cnrm = pd.read_csv(cnrm_src).rename(columns={'Longitude':'lon', 'Latitude':'lat', 'Date':'time', 'ID':'track_id'})
hybrids = {
    hybrid_key_renames[src.split('/')[-2]] : pd.read_csv(src, index_col=0).rename(columns={'ISO_TIME':'time', 'LAT':'lat', 'LON':'lon', 'WS':'ws', 'TRACK_ID':'track_id', 'HAVERSINE':'haversine'})
    for src in hybrid_fnames
}

trackers = {
    'ibtracs': ibtracs, 
    'owz': owz, 
    'track': track, 
    'uz': uz, 
    'cnrm': cnrm, 
    **hybrids
}
trackers.keys()

In [ ]:
# convert iso timed to pandas datetime format
for key, value in trackers.items():
    if key == 'owz' or key == 'uz':
        trackers[key]['time'] = pd.to_datetime(trackers[key][['year','month','day','hour']])
    else:
        trackers[key]['time'] = pd.to_datetime(trackers[key]['time'])

In [ ]:
# separate ENP and WNP basins
for key, value in trackers.items():
    trackers[key] = filter_basin(trackers[key])

In [ ]:
# get intersection of all dates to be sure that each timesteps has been seen by EVERY tracker
dates_df = pd.read_csv(os.path.join(inference_src, 'dates.csv'))
dates = pd.to_datetime(dates_df['dates'].to_numpy())
# get only dates within the test set years
dates = pd.to_datetime([date for date in dates if date.year in years and date.month in months])

In [ ]:
# filter the tracks according to the procedure
for key, value in trackers.items():
    trackers[key] = filter_tracks(trackers[key], dates, lat_range, lon_range, min_track_count, remove_atlantic, remove_short_tracks)

# Dynamicopy Track Matching

In [ ]:
# compute pod and far for each tracker
trackers_matches = {}
trackers_res = {}
for key, value in trackers.items():
    if key == 'ibtracs': continue
    trackers_matches[key], trackers_res[key] = compute_pod_and_far(dynamicopy, trackers[key], key, trackers['ibtracs'], max_track_distance_matching, print_results=False)

trackers_matches_enp = {}
trackers_res_enp = {}
for key, value in trackers.items():
    if key == 'ibtracs': continue
    trk = trackers[key]
    ibt = trackers['ibtracs']
    trk = trk[trk['basin']=='ENP']
    ibt = ibt[ibt['basin']=='ENP']
    trackers_matches_enp[key], trackers_res_enp[key] = compute_pod_and_far(dynamicopy, trk, key, ibt, max_track_distance_matching, print_results=False)

trackers_matches_wnp = {}
trackers_res_wnp = {}
for key, value in trackers.items():
    if key == 'ibtracs': continue
    trk = trackers[key]
    ibt = trackers['ibtracs']
    trk = trk[trk['basin']=='WNP']
    ibt = ibt[ibt['basin']=='WNP']
    trackers_matches_wnp[key], trackers_res_wnp[key] = compute_pod_and_far(dynamicopy, trk, key, ibt, max_track_distance_matching, print_results=False)


In [ ]:
algo_results = pd.concat([res for res in trackers_res.values()]).reset_index(drop=True)
algo_results['pod'] = algo_results['pod'] * 100
algo_results['far'] = algo_results['far'] * 100

algo_results_enp = pd.concat([res for res in trackers_res_enp.values()]).reset_index(drop=True)
algo_results_enp['pod'] = algo_results_enp['pod'] * 100
algo_results_enp['far'] = algo_results_enp['far'] * 100

algo_results_wnp = pd.concat([res for res in trackers_res_wnp.values()]).reset_index(drop=True)
algo_results_wnp['pod'] = algo_results_wnp['pod'] * 100
algo_results_wnp['far'] = algo_results_wnp['far'] * 100


In [ ]:
outfile = os.path.join(out_dir, 'pod_far.png') if save_figures else None
plot_pod_and_far_multi_trackers(algo_results, trackers, '(1980-2019)', 0, outfile)

outfile = os.path.join(out_dir, 'pod_far_ENP.png') if save_figures else None
plot_pod_and_far_multi_trackers(algo_results_enp, trackers, '(1980-2019)', 0, outfile)

outfile = os.path.join(out_dir, 'pod_far_WNP.png') if save_figures else None
plot_pod_and_far_multi_trackers(algo_results_wnp, trackers, '(1980-2019)', 0, outfile)

# Spatial Distribution

Plot the spatial distribution of TC detections over latitude and longitude

In [ ]:
trackers.keys()

In [ ]:
plt.figure(figsize=(15,8))

for key in trackers.keys():
    linewidth = 2.0 if key not in ['owz', 'track', 'uz', 'cnrm'] else 1.0
    freq_x, bin_edges_x = np.histogram(trackers[key]['lat'].to_numpy(), range=lat_range, bins=lat_range[1])
    plt.plot(bin_edges_x[:-1], freq_x, label=key.upper(), drawstyle='steps', linewidth=linewidth)

plt.xlabel('Latitudes', fontdict={'weight':'bold'})
plt.ylabel('Frequencies', fontdict={'weight':'bold'})
plt.xlim(*lat_range)
plt.title(f'Latitude distributions of TC detections')
plt.legend()

if save_figures:
    plt.savefig(os.path.join(out_dir, 'spatial_dist_lat.png'), dpi=300)
else:
    plt.show()

In [ ]:
plt.figure(figsize=(15,5))

for key in trackers.keys():
    linewidth = 2.0 if key not in ['owz', 'track', 'uz', 'cnrm'] else 1.0
    freq_x, bin_edges_x = np.histogram(((trackers[key]['lon'] + 360) % 360).to_numpy(), range=lon_range, bins=lon_range[1]-lon_range[0])
    plt.plot(bin_edges_x[:-1], freq_x, label=key.upper(), drawstyle='steps', linewidth=linewidth)

plt.xlabel('Longitudes', fontdict={'weight':'bold'})
plt.ylabel('Frequencies', fontdict={'weight':'bold'})
plt.xlim(*lon_range)
plt.title(f'Longitude distributions of TC detections')
plt.legend()

if save_figures:
    plt.savefig(os.path.join(out_dir, 'spatial_dist_lon.png'), dpi=300)
else:
    plt.show()

# Tropical Storm Track Density

It is computed as the storm transit per month on each point in the domain

In [ ]:
lats = np.linspace(0, 70, 281)[::-1]
lons = np.linspace(100, 320, 881)

In [ ]:
bins = [14, 44] # 70°, 220° -> for 5°x5° grid bins = [14, 44] - for 2°x2° grid, bins = [35, 110]
norm_month = False

trackers_storm_transits = {}
for key, value in trackers.items():
    trackers_storm_transits[key] = compute_storm_transits(value, bins, lats, lons, norm_month)

vmin = np.nanmin([transit for transit in trackers_storm_transits.values()])
vmax = np.nanmax([transit for transit in trackers_storm_transits.values()])

for key, value in trackers_storm_transits.items():
    plot_storm_track_density(value, f'{key}', vmin, vmax, remove_atlantic, lat_range, lon_range)
    plt.savefig(os.path.join(out_dir, f'{key}_storm_transits.png'), dpi=300) if save_figures else plt.show()

# Track Durations

In [ ]:
plt.figure(figsize=(10,6))

trackers_track_durations = {}
trackers_track_durations_short = {}
for key, value in trackers.items():
    linewidth = 2.0 if key not in ['owz', 'track', 'uz', 'cnrm'] else 1.0
    track_durations = value.track_id.value_counts().to_numpy()//4
    bins = (track_durations).max()
    freq_x, bin_edges_x = np.histogram(track_durations, bins=bins, range=(3,bins+3))
    plt.plot(bin_edges_x[:-3], freq_x[:-2], label=key.upper(), drawstyle='steps', linewidth=linewidth)
    trackers_track_durations[key] = track_durations
    trackers_track_durations_short[key] = freq_x.max()

plt.xlabel('Track duration (days)', fontdict={'weight':'bold'})
plt.title(f'Track Duration')
plt.legend()

plt.savefig(os.path.join(out_dir, 'track_duration_days.png'), dpi=300) if save_figures else plt.show()

# Seasonality

Plot number of TC tracks for every month of year

- Maybe create a plot for each Tracker
- Maybe add only False Alarms and Hits (as Bourdin)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for key, value in trackers.items():
    linewidth = 2.0 if key not in ['owz', 'track', 'uz', 'cnrm'] else 1.0
    freq_x, bin_edges_x = np.histogram(value.time.dt.month, bins=12, range=[1,13])
    plt.plot(bin_edges_x[:-1], freq_x, label=key.upper(), drawstyle='steps', linewidth=linewidth)

plt.xlabel('Month')
plt.ylabel('Freq')
ax.set_xticks(np.arange(1, 13, 1))
ax.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
plt.title(f'Seasonal Tropical Cyclone distribution')
plt.legend()

plt.savefig(os.path.join(out_dir, 'seasonal_tc_dist.png'), dpi=300) if save_figures else plt.show()

# Interannual Variability

Plot the number of TC tracks for every year

In [ ]:
# compute interannual variabilities

for key, value in trackers.items():
    value['year'] = value['time'].dt.year

years_df = pd.DataFrame(data={'year':years})

trackers_iav_enp, trackers_iav_wnp = {}, {}
for key, value in trackers.items():
    trackers_iav_enp[key] = pd.merge(value[value['basin']=='ENP'].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    trackers_iav_wnp[key] = pd.merge(value[value['basin']=='WNP'].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()

In [ ]:
trackers_iav_enp_pearsonr = {
    key: pearsonr(value, trackers_iav_enp['ibtracs'])
    for key, value in trackers_iav_enp.items()
}
trackers_iav_enp_pearsonr_round = {
    key: np.round(value.statistic, 2)
    for key, value in trackers_iav_enp_pearsonr.items()
}
trackers_iav_enp_pearsonr_signif = {
    key: signif(value.pvalue, 2)
    for key, value in trackers_iav_enp_pearsonr.items()
}

In [ ]:
fig, ax = plt.subplots(figsize=(15,6))

plt.title(f'Interannual Variability (1980-2019, Aug) on ENP basin')
for key in trackers.keys():
    linewidth = 2.0 if key not in ['owz', 'track', 'uz', 'cnrm'] else 1.0
    tr_iav_enp = trackers_iav_enp[key]
    tr_pearson = trackers_iav_enp_pearsonr_round[key]
    tr_signif = trackers_iav_enp_pearsonr_signif[key]
    plt.plot(np.arange(len(years)), tr_iav_enp, label=f'{key.upper()} (pearson = {tr_pearson}, p = {tr_signif})', linewidth=linewidth)

ax.set_xticks(np.arange(len(years)))
ax.set_xticklabels(years, rotation=45)

ax.set_xlabel('Year', fontdict={'weight':'bold', 'size':'14'})
ax.set_ylabel('Number of TCs', fontdict={'weight':'bold', 'size':'14'})

plt.ylim(0, None)

box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
ax.legend(loc='upper left', markerscale=8, edgecolor='gray', framealpha=1, ncol=3, bbox_to_anchor=(0.07 ,-0.16))

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'interannual_variability_enp_basin.png'), dpi=300) if save_figures else plt.show()

In [ ]:
trackers_iav_wnp_pearsonr = {
    key: pearsonr(value, trackers_iav_wnp['ibtracs'])
    for key, value in trackers_iav_wnp.items()
}
trackers_iav_wnp_pearsonr_round = {
    key: np.round(value.statistic, 2)
    for key, value in trackers_iav_wnp_pearsonr.items()
}
trackers_iav_wnp_pearsonr_signif = {
    key: signif(value.pvalue, 2)
    for key, value in trackers_iav_wnp_pearsonr.items()
}

In [ ]:
fig, ax = plt.subplots(figsize=(15,6))

plt.title(f'Interannual Variability (1980-2019, Aug) on WNP basin')

for key in trackers.keys():
    linewidth = 2.0 if key not in ['owz', 'track', 'uz', 'cnrm'] else 1.0
    tr_iav_wnp = trackers_iav_wnp[key]
    tr_pearson = trackers_iav_wnp_pearsonr_round[key]
    tr_signif = trackers_iav_wnp_pearsonr_signif[key]
    plt.plot(np.arange(len(years)), tr_iav_wnp, label=f'{key.upper()} (pearson = {tr_pearson}, p = {tr_signif})', linewidth=linewidth)

ax.set_xticks(np.arange(len(years)))
ax.set_xticklabels(years, rotation=45)

ax.set_xlabel('Year', fontdict={'weight':'bold', 'size':'14'})
ax.set_ylabel('Number of TCs', fontdict={'weight':'bold', 'size':'14'})

plt.ylim(0, None)

box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
ax.legend(loc='upper left', markerscale=8, edgecolor='gray', framealpha=1, ncol=3, bbox_to_anchor=(0.08 ,-0.16))

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'interannual_variability_wnp_basin.png'), dpi=300) if save_figures else plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15,6))

plt.title(f'Interannual Variability (1980-2019, Aug) on WNP basin')

for key in trackers.keys():
    if key in ['owz', 'track', 'uz', 'cnrm']: continue
    linewidth = 2.0 if key not in ['owz', 'track', 'uz', 'cnrm'] else 1.0
    tr_iav_wnp = trackers_iav_wnp[key]
    tr_pearson = trackers_iav_wnp_pearsonr_round[key]
    tr_signif = trackers_iav_wnp_pearsonr_signif[key]
    plt.plot(np.arange(len(years)), tr_iav_wnp, label=f'{key.upper()} (pearson = {tr_pearson}, p = {tr_signif})', linewidth=linewidth)

ax.set_xticks(np.arange(len(years)))
ax.set_xticklabels(years, rotation=45)

ax.set_xlabel('Year', fontdict={'weight':'bold', 'size':'14'})
ax.set_ylabel('Number of TCs', fontdict={'weight':'bold', 'size':'14'})

plt.ylim(0, None)

box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
ax.legend(loc='upper left', markerscale=8, edgecolor='gray', framealpha=1, ncol=3, bbox_to_anchor=(0.08 ,-0.16))

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'interannual_variability_wnp_basin_only_ml.png'), dpi=300) if save_figures else plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15,6))

plt.title(f'Interannual Variability (1980-2019, Aug) on ENP basin')
for key in trackers.keys():
    if key in ['owz', 'track', 'uz', 'cnrm']: continue
    linewidth = 2.0 if key not in ['owz', 'track', 'uz', 'cnrm'] else 1.0
    tr_iav_enp = trackers_iav_enp[key]
    tr_pearson = trackers_iav_enp_pearsonr_round[key]
    tr_signif = trackers_iav_enp_pearsonr_signif[key]
    plt.plot(np.arange(len(years)), tr_iav_enp, label=f'{key.upper()} (pearson = {tr_pearson}, p = {tr_signif})', linewidth=linewidth)

ax.set_xticks(np.arange(len(years)))
ax.set_xticklabels(years, rotation=45)

ax.set_xlabel('Year', fontdict={'weight':'bold', 'size':'14'})
ax.set_ylabel('Number of TCs', fontdict={'weight':'bold', 'size':'14'})

plt.ylim(0, None)

box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
ax.legend(loc='upper left', markerscale=8, edgecolor='gray', framealpha=1, ncol=3, bbox_to_anchor=(0.07 ,-0.16))

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'interannual_variability_enp_basin_only_ml.png'), dpi=300) if save_figures else plt.show()

In [ ]:
ml_tracker = list(trackers.keys())[-1]
sota_tracker = 'track'
year = 2018

In [ ]:
outfile = os.path.join(out_dir, f'{ml_tracker}_{year}_tracks.png') if save_figures else None

det_tracks = trackers[ml_tracker][trackers[ml_tracker]['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'})
obs_tracks = trackers['ibtracs'][trackers['ibtracs']['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'})
det_tracks['LON'] = (det_tracks['LON'] + 360) % 360
obs_tracks['LON'] = (obs_tracks['LON'] + 360) % 360
plot_tracks(det_tracks = det_tracks, 
            obs_tracks = obs_tracks, 
            lat_range = lat_range, 
            lon_range = lon_range, 
            outfile = outfile
            )

In [ ]:
outfile = os.path.join(out_dir, f'{sota_tracker}_{year}_tracks.png') if save_figures else None

det_tracks = trackers[sota_tracker][trackers[sota_tracker]['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'})
obs_tracks = trackers['ibtracs'][trackers['ibtracs']['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'})
det_tracks['LON'] = (det_tracks['LON'] + 360) % 360
obs_tracks['LON'] = (obs_tracks['LON'] + 360) % 360
plot_tracks(det_tracks=det_tracks, 
            obs_tracks=obs_tracks, 
            lat_range=lat_range, 
            lon_range=lon_range, 
            outfile = outfile, 
            )

# Classification Probabilities for TPs, FPs and FNs

In [ ]:
for key, value in trackers.items():
    if key in ['ibtracs', 'track', 'owz', 'uz', 'cnrm']: continue

    D = trackers[key]
    M = trackers_matches[key]
    
    FP_track_ids = list(set(D['track_id'].unique()).difference(set(M[f'id_{key}'].unique())))
    TP_track_ids = list(set(M[f'id_{key}'].unique()))
    
    FP_probs = []
    for track_id in FP_track_ids:
        probs = list(D[D['track_id'] == track_id]['PROB'])
        FP_probs += probs
    FP_probs = np.asarray(FP_probs)
    
    TP_probs = []
    for track_id in TP_track_ids:
        probs = list(D[D['track_id'] == track_id]['PROB'])
        TP_probs += probs
    TP_probs = np.asarray(TP_probs)
    
    print(f'Tracker {key}')
    print(f'   False Positives: MIN {FP_probs.min()}, MAX {FP_probs.max()}, MEAN {FP_probs.mean()}, STD {FP_probs.std()}')
    print(f'   True Positives: MIN {TP_probs.min()}, MAX {TP_probs.max()}, MEAN {TP_probs.mean()}, STD {TP_probs.std()}')

# Merge Short Tracks

In [ ]:
# the merging of duplicates is in-place
trackers_overlap = {}
for key, value in trackers.items():
    if key == 'ibtracs': continue
    trackers_overlap[key] = dynamicopy.merge_duplicates(value, trackers['ibtracs'])

In [ ]:
trackers_track_durations_merge = {}
trackers_track_durations_merge_short = {}
for key, value in trackers.items():
    if key == 'ibtracs':
        trackers_track_durations_merge_short[key] = np.nan
        track_durations = value.track_id.value_counts().to_numpy()//4
        trackers_track_durations_merge[key] = track_durations
    else:
        track_durations = value.track_id.value_counts().to_numpy()//4
        bins = (track_durations).max()
        freq_x, bin_edges_x = np.histogram(track_durations, bins=bins, range=(3,bins+3))
        trackers_track_durations_merge[key] = track_durations
        trackers_track_durations_merge_short[key] = freq_x.max()

# Put all Together in a Table

In [ ]:
table = pd.DataFrame()
for key in trackers.keys():
    # if key == 'ibtracs': continue
    try:
        pod = np.round(algo_results[algo_results['algo'] == key.upper()]['pod'].to_numpy()[0], 2)
        far = np.round(algo_results[algo_results['algo'] == key.upper()]['far'].to_numpy()[0], 2)
    except:
        pod = None
        far = None
    pearson_enp = trackers_iav_enp_pearsonr_round[key]
    pvalue_enp = trackers_iav_enp_pearsonr_signif[key]
    pearson_wnp = trackers_iav_wnp_pearsonr_round[key]
    pvalue_wnp = trackers_iav_wnp_pearsonr_signif[key]
    dur_max = trackers_track_durations[key].max()
    dur_short = trackers_track_durations_short[key]
    dur_merge_max = trackers_track_durations_merge[key].max()
    dur_merge_short = trackers_track_durations_merge_short[key]
    table = pd.concat((table, pd.DataFrame(data={
        'tracker': [key.upper()], 
        'pod' : [pod], 
        'far' : [far], 
        'pearson_enp' : [pearson_enp], 
        'pvalue_enp' : [pvalue_enp], 
        'pearson_wnp' : [pearson_wnp], 
        'pvalue_wnp' : [pvalue_wnp], 
        'dur_max' : [dur_max], 
        'dur_short' : [dur_short], 
        'dur_merge_max': [dur_merge_max], 
        'dur_merge_short': [dur_merge_short], 
    }))).reset_index(drop=True)

table.to_csv(os.path.join(out_dir, 'results_table.csv')) if save_figures else None
table